# 12 Tuned Word+Char TF-IDF and Probability Ensemble

Tune only the Logistic Regression C parameter for the word+char TF-IDF A/B swap model, then ensemble it with the current best A/B swap TF-IDF model.

## 1. Imports and Path Setup

Set up libraries, constants, project paths, and output folders.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

RANDOM_STATE = 42
LABELS = [0, 1, 2]
LABEL_NAME_MAP = {0: 'A_win', 1: 'B_win', 2: 'tie'}
PROB_COLUMNS = ['prob_A_win', 'prob_B_win', 'prob_tie']
SUB_PROB_COLUMNS = ['winner_model_a', 'winner_model_b', 'winner_tie']
SUB_COLUMNS = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
ENSEMBLE_BASELINE_LOG_LOSS = 1.077993

current_dir = Path.cwd().resolve()
if current_dir.name.lower() == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

processed_data_dir = project_root / 'data' / 'processed'
logs_dir = project_root / 'outputs' / 'logs'
oof_dir = project_root / 'outputs' / 'oof'
submissions_dir = project_root / 'outputs' / 'submissions'

for output_dir in [logs_dir, oof_dir, submissions_dir]:
    output_dir.mkdir(parents=True, exist_ok=True)

train_path = processed_data_dir / 'train_eda.csv'
test_path = processed_data_dir / 'test_eda.csv'
ab_swap_valid_path = oof_dir / 'tfidf_lr_ab_swap_valid_predictions.csv'
ab_swap_submission_path = submissions_dir / 'tfidf_lr_ab_swap_avg_submission.csv'

c_tuning_results_path = logs_dir / 'word_char_c_tuning_results.csv'
tuned_valid_predictions_path = oof_dir / 'tfidf_word_char_ab_swap_tuned_valid_predictions.csv'
tuned_submission_path = submissions_dir / 'tfidf_word_char_ab_swap_tuned_submission.csv'
ensemble_results_path = logs_dir / 'tuned_word_char_ensemble_results.csv'
ensemble_submission_path = submissions_dir / 'ensemble_ab_swap_tuned_word_char_submission.csv'
experiment_results_path = logs_dir / 'experiment_results.csv'

print(f'Project root: {project_root}')
for path in [train_path, test_path, ab_swap_valid_path, ab_swap_submission_path]:
    print(f'{path.exists()} -> {path}')

Project root: D:\LLM_Classification_finetuning
True -> D:\LLM_Classification_finetuning\data\processed\train_eda.csv
True -> D:\LLM_Classification_finetuning\data\processed\test_eda.csv
True -> D:\LLM_Classification_finetuning\outputs\oof\tfidf_lr_ab_swap_valid_predictions.csv
True -> D:\LLM_Classification_finetuning\outputs\submissions\tfidf_lr_ab_swap_avg_submission.csv


## 2. Read Processed Data

Read processed train and test files. Raw data is not used.

In [2]:
train = pd.read_csv(train_path, encoding='utf-8-sig')
test = pd.read_csv(test_path, encoding='utf-8-sig')

required_train_columns = {'id', 'label', 'label_name', 'prompt_clean', 'response_a_clean', 'response_b_clean'}
required_test_columns = {'id', 'prompt_clean', 'response_a_clean', 'response_b_clean'}

missing_train_columns = sorted(required_train_columns - set(train.columns))
missing_test_columns = sorted(required_test_columns - set(test.columns))

if missing_train_columns:
    raise ValueError(f'train_eda.csv missing columns: {missing_train_columns}')
if missing_test_columns:
    raise ValueError(f'test_eda.csv missing columns: {missing_test_columns}')

train['label'] = train['label'].astype(int)

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print('\nLabel counts:')
display(train['label_name'].value_counts().reindex(['A_win', 'B_win', 'tie']))

train shape: (57477, 20)
test shape: (3, 12)

Label counts:


label_name
A_win    20064
B_win    19652
tie      17761
Name: count, dtype: int64

## 3. Helper Functions

Define text construction, A/B swapping, sparse feature construction, probability alignment, and metric helpers.

In [3]:
def make_text_input(df):
    prompt = df['prompt_clean'].fillna('').astype(str)
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    return (
        'Prompt:\n' + prompt
        + '\n\nResponse A:\n' + response_a
        + '\n\nResponse B:\n' + response_b
    )


def swap_ab_dataframe(df):
    swapped = df.copy()
    swapped['response_a_clean'] = df['response_b_clean'].values
    swapped['response_b_clean'] = df['response_a_clean'].values

    if 'label' in swapped.columns:
        swapped['label'] = df['label'].map({0: 1, 1: 0, 2: 2}).astype(int).values
        swapped['label_name'] = swapped['label'].map(LABEL_NAME_MAP)

    return swapped


def align_probabilities(probabilities, classes, labels=LABELS):
    aligned = np.zeros((probabilities.shape[0], len(labels)), dtype=float)
    class_to_position = {int(label): idx for idx, label in enumerate(classes)}

    for output_position, label in enumerate(labels):
        if label in class_to_position:
            aligned[:, output_position] = probabilities[:, class_to_position[label]]

    return aligned


def transform_word_char(word_vectorizer, char_vectorizer, text_series):
    word_features = word_vectorizer.transform(text_series)
    char_features = char_vectorizer.transform(text_series)
    return hstack([word_features, char_features], format='csr')


def evaluate_probs(y_true, probabilities):
    predictions = np.argmax(probabilities, axis=1)
    return {
        'valid_log_loss': log_loss(y_true, probabilities, labels=LABELS),
        'valid_accuracy': accuracy_score(y_true, predictions),
        'valid_macro_f1': f1_score(y_true, predictions, average='macro'),
    }


print('Helper functions ready.')

Helper functions ready.


## 4. Fixed Split and A/B Swap Augmentation

Use the same fixed train/valid split as previous experiments, then augment only the training split.

In [4]:
train_split, valid_split = train_test_split(
    train,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train['label'],
)

train_split = train_split.reset_index(drop=True)
valid_split = valid_split.reset_index(drop=True)

swapped_train = swap_ab_dataframe(train_split)
train_aug = pd.concat([train_split, swapped_train], ignore_index=True)
train_aug = train_aug.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

train_aug['text_input'] = make_text_input(train_aug)
valid_split['text_input'] = make_text_input(valid_split)
valid_swapped = swap_ab_dataframe(valid_split.copy())
valid_swapped['text_input'] = make_text_input(valid_swapped)
test['text_input'] = make_text_input(test)

y_train = train_aug['label'].astype(int)
y_valid = valid_split['label'].astype(int).to_numpy()

print(f'train_split shape: {train_split.shape}')
print(f'valid_split shape: {valid_split.shape}')
print(f'train_aug shape: {train_aug.shape}')
print('\nAugmented train label counts:')
display(train_aug['label_name'].value_counts().reindex(['A_win', 'B_win', 'tie']))

train_split shape: (45981, 20)
valid_split shape: (11496, 21)
train_aug shape: (91962, 21)

Augmented train label counts:


label_name
A_win    31772
B_win    31772
tie      28418
Name: count, dtype: int64

## 5. Fit Word and Char TF-IDF Features

Fit word-level and char-level TF-IDF vectorizers once, then reuse the feature matrices for all C values.

In [5]:
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    max_features=50000,
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_word = word_vectorizer.fit_transform(train_aug['text_input'])
X_train_char = char_vectorizer.fit_transform(train_aug['text_input'])
X_train_features = hstack([X_train_word, X_train_char], format='csr')

X_valid_features = transform_word_char(word_vectorizer, char_vectorizer, valid_split['text_input'])
X_valid_swapped_features = transform_word_char(word_vectorizer, char_vectorizer, valid_swapped['text_input'])

print(f'X_train_features shape: {X_train_features.shape}')
print(f'X_valid_features shape: {X_valid_features.shape}')
print(f'X_valid_swapped_features shape: {X_valid_swapped_features.shape}')

X_train_features shape: (91962, 150000)
X_valid_features shape: (11496, 150000)
X_valid_swapped_features shape: (11496, 150000)


## 6. Tune Logistic Regression C

Train one Logistic Regression model per C value and evaluate each with validation swap-test averaging.

In [6]:
C_list = [0.02, 0.03, 0.05, 0.08, 0.1, 0.15]
c_tuning_results = []
best_word_char_model = None
best_word_char_probs = None
best_word_char_C = None
best_word_char_valid_log_loss = np.inf

for C in C_list:
    print(f'Training word+char Logistic Regression with C={C}')
    model = LogisticRegression(
        C=C,
        max_iter=1000,
        solver='saga',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_features, y_train)

    probs_original = align_probabilities(model.predict_proba(X_valid_features), model.classes_, LABELS)
    probs_swapped = align_probabilities(model.predict_proba(X_valid_swapped_features), model.classes_, LABELS)
    mapped_probs_swapped = np.column_stack([
        probs_swapped[:, 1],
        probs_swapped[:, 0],
        probs_swapped[:, 2],
    ])
    avg_probs = 0.5 * probs_original + 0.5 * mapped_probs_swapped
    metrics = evaluate_probs(y_valid, avg_probs)
    metrics['C'] = C
    c_tuning_results.append(metrics)

    print(
        f"C={C}: log_loss={metrics['valid_log_loss']:.6f}, "
        f"accuracy={metrics['valid_accuracy']:.6f}, "
        f"macro_f1={metrics['valid_macro_f1']:.6f}"
    )

    if metrics['valid_log_loss'] < best_word_char_valid_log_loss:
        best_word_char_valid_log_loss = metrics['valid_log_loss']
        best_word_char_C = C
        best_word_char_model = model
        best_word_char_probs = avg_probs

c_tuning_results_df = pd.DataFrame(c_tuning_results).sort_values('valid_log_loss').reset_index(drop=True)
c_tuning_results_df.to_csv(c_tuning_results_path, index=False, encoding='utf-8-sig')

best_word_char_valid_accuracy = float(c_tuning_results_df.loc[0, 'valid_accuracy'])
best_word_char_valid_macro_f1 = float(c_tuning_results_df.loc[0, 'valid_macro_f1'])

print(f'Saved C tuning results: {c_tuning_results_path}')
display(c_tuning_results_df)
print(f'best_word_char_C: {best_word_char_C}')
print(f'best_word_char_valid_log_loss: {best_word_char_valid_log_loss:.6f}')

Training word+char Logistic Regression with C=0.02
C=0.02: log_loss=1.079915, accuracy=0.395268, macro_f1=0.389189
Training word+char Logistic Regression with C=0.03
C=0.03: log_loss=1.079002, accuracy=0.395268, macro_f1=0.389145
Training word+char Logistic Regression with C=0.05
C=0.05: log_loss=1.078429, accuracy=0.395007, macro_f1=0.388729
Training word+char Logistic Regression with C=0.08
C=0.08: log_loss=1.078739, accuracy=0.393006, macro_f1=0.386314
Training word+char Logistic Regression with C=0.1
C=0.1: log_loss=1.079273, accuracy=0.392136, macro_f1=0.385576
Training word+char Logistic Regression with C=0.15
C=0.15: log_loss=1.081122, accuracy=0.389962, macro_f1=0.383514
Saved C tuning results: D:\LLM_Classification_finetuning\outputs\logs\word_char_c_tuning_results.csv


,valid_log_loss,valid_accuracy,valid_macro_f1,C
0,1.078429,0.395007,0.388729,0.05
1,1.078739,0.393006,0.386314,0.08
2,1.079002,0.395268,0.389145,0.03
3,1.079273,0.392136,0.385576,0.10
4,1.079915,0.395268,0.389189,0.02
5,1.081122,0.389962,0.383514,0.15


best_word_char_C: 0.05
best_word_char_valid_log_loss: 1.078429


## 7. Save Tuned Word+Char Validation Predictions

Save the best tuned word+char validation probabilities for later ensemble experiments.

In [7]:
best_word_char_pred = np.argmax(best_word_char_probs, axis=1)

tuned_valid_predictions = valid_split.copy()
tuned_valid_predictions['pred_label'] = best_word_char_pred
tuned_valid_predictions['pred_label_name'] = tuned_valid_predictions['pred_label'].map(LABEL_NAME_MAP)
tuned_valid_predictions['prob_A_win'] = best_word_char_probs[:, 0]
tuned_valid_predictions['prob_B_win'] = best_word_char_probs[:, 1]
tuned_valid_predictions['prob_tie'] = best_word_char_probs[:, 2]
tuned_valid_predictions['best_C'] = best_word_char_C

valid_prediction_columns = [
    'id',
    'label',
    'label_name',
    'pred_label',
    'pred_label_name',
    'prob_A_win',
    'prob_B_win',
    'prob_tie',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'best_C',
]

for optional_column in ['response_a_char_len', 'response_b_char_len', 'response_len_diff']:
    if optional_column in tuned_valid_predictions.columns:
        valid_prediction_columns.append(optional_column)

tuned_valid_predictions[valid_prediction_columns].to_csv(
    tuned_valid_predictions_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Saved tuned validation predictions: {tuned_valid_predictions_path}')
display(tuned_valid_predictions[valid_prediction_columns].head())

Saved tuned validation predictions: D:\LLM_Classification_finetuning\outputs\oof\tfidf_word_char_ab_swap_tuned_valid_predictions.csv


,id,label,label_name,pred_label,pred_label_name,prob_A_win,prob_B_win,prob_tie,prompt_clean,response_a_clean,response_b_clean,best_C,response_a_char_len,response_b_char_len,response_len_diff
0,2098233919,0,A_win,0,A_win,0.364476,0.364476,0.271048,"How many tires would you estimate are in the city of Detroit, Michigan? Answer in no more than 3 sentences with your reason.\nWhat about the fact that cars ...","I estimate there are over 1 million tires in the city of Detroit, Michigan. As a major metropolitan area, Detroit has hundreds of thousands of registered ve...","It is difficult to provide an exact estimate without specific data, but we can make a rough estimation. Considering the population of Detroit and assuming a...",0.05,795,833,-38
1,2789033728,1,B_win,1,B_win,0.360700,0.360903,0.278397,Write KPI objectives and KPI strategies for Move all companies ISO processes to new ERP,Here are some sample KPI objectives and strategies for moving all company ISO processes to a new ERP system:\n\nKPI Objectives:\n- Complete data migration f...,KPI Objectives:\n1. Successfully migrate all companies' ISO processes to the new ERP system.\n2. Minimize disruption to business operations during the migra...,0.05,1504,1599,-95
2,4198930172,2,tie,1,B_win,0.352754,0.352872,0.294374,Name all the skills of Disco Elysium and format your answer as follows:\nName of skill - Relative rating of usefulness from 0 to 100 - Relative rating of in...,Skills of Disco Elysium:\n\n1. Persuasion - 80\/100 - 80\/100\n2. Intimidation - 80\/100 - 80\/100\n3. Investigation - 80\/100 - 80\/100\n4. Stealth...,"Sure, here are the skills in Disco Elysium, along with their relative ratings of usefulness and interestingness:\n\n1. Intelligence - 90% usefulness, 50% in...",0.05,1019,1340,-321
3,2396441388,1,B_win,0,A_win,0.340158,0.340158,0.319684,how do you do patch a hole in drywall?,"Patching a hole in drywall involves a few steps. Here's a general guide:\n\n1. Gather materials: You will need a piece of drywall, drywall mud or joint comp...","Patching a hole in drywall involves several steps, including gathering the necessary tools and materials, preparing the area around the hole, applying a pat...",0.05,1644,2188,-544
4,3592485435,2,tie,1,B_win,0.337078,0.337588,0.325335,"John places his slice of pie in the refrigerator shared by his coworkers, then goes into his office and closes the door. He cannot see the refrigerator. Sus...","The refrigerator cannot communicate with John directly. However, if John notices that his slice of pie is missing, he might conclude that it has been eaten ...",The refrigerator does not have the ability to communicate or tell John anything. It is an inanimate object. John may notice that his pie is missing when he ...,0.05,319,171,148


## 8. Save Tuned Word+Char Submission

Use the best tuned word+char model to predict test probabilities with swap-test averaging.

In [8]:
test_swapped = swap_ab_dataframe(test.copy())
test_swapped['text_input'] = make_text_input(test_swapped)

X_test_features = transform_word_char(word_vectorizer, char_vectorizer, test['text_input'])
X_test_swapped_features = transform_word_char(word_vectorizer, char_vectorizer, test_swapped['text_input'])

test_probs_original = align_probabilities(best_word_char_model.predict_proba(X_test_features), best_word_char_model.classes_, LABELS)
test_probs_swapped = align_probabilities(best_word_char_model.predict_proba(X_test_swapped_features), best_word_char_model.classes_, LABELS)
mapped_test_probs_swapped = np.column_stack([
    test_probs_swapped[:, 1],
    test_probs_swapped[:, 0],
    test_probs_swapped[:, 2],
])
tuned_test_probs = 0.5 * test_probs_original + 0.5 * mapped_test_probs_swapped

tuned_submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': tuned_test_probs[:, 0],
    'winner_model_b': tuned_test_probs[:, 1],
    'winner_tie': tuned_test_probs[:, 2],
})
tuned_submission.to_csv(tuned_submission_path, index=False, encoding='utf-8-sig')

print(f'Saved tuned word+char submission: {tuned_submission_path}')
print(f'tuned_submission shape: {tuned_submission.shape}')
print(f'probability sums close to 1: {np.allclose(tuned_submission[SUB_PROB_COLUMNS].sum(axis=1), 1.0, atol=1e-6)}')
print(f'has NaN: {tuned_submission.isna().any().any()}')
display(tuned_submission.head())

Saved tuned word+char submission: D:\LLM_Classification_finetuning\outputs\submissions\tfidf_word_char_ab_swap_tuned_submission.csv
tuned_submission shape: (3, 4)
probability sums close to 1: True
has NaN: False


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.241039,0.241039,0.517921
1,211333,0.351005,0.350628,0.298367
2,1233961,0.395797,0.395797,0.208405


## 9. Re-Ensemble with Current Best A/B Swap Model

Read current best A/B swap predictions and tuned word+char predictions, align by id, and search ensemble weights.

In [9]:
ab_swap_valid = pd.read_csv(ab_swap_valid_path, encoding='utf-8-sig')
tuned_word_char_valid = pd.read_csv(tuned_valid_predictions_path, encoding='utf-8-sig')

required_valid_columns = {'id', 'label', 'prob_A_win', 'prob_B_win', 'prob_tie'}
missing_ab_columns = sorted(required_valid_columns - set(ab_swap_valid.columns))
missing_wc_columns = sorted(required_valid_columns - set(tuned_word_char_valid.columns))
if missing_ab_columns:
    raise ValueError(f'ab swap valid predictions missing columns: {missing_ab_columns}')
if missing_wc_columns:
    raise ValueError(f'tuned word char valid predictions missing columns: {missing_wc_columns}')

merged_valid = ab_swap_valid[['id', 'label'] + PROB_COLUMNS].merge(
    tuned_word_char_valid[['id', 'label'] + PROB_COLUMNS],
    on='id',
    suffixes=('_ab_swap', '_word_char'),
    how='inner',
)

if len(merged_valid) != len(ab_swap_valid) or len(merged_valid) != len(tuned_word_char_valid):
    raise ValueError('Validation prediction files are not perfectly aligned by id.')
if not (merged_valid['label_ab_swap'].to_numpy() == merged_valid['label_word_char'].to_numpy()).all():
    raise ValueError('Validation labels do not match after id alignment.')

ensemble_y_valid = merged_valid['label_ab_swap'].astype(int).to_numpy()
ab_swap_probs = merged_valid[[f'{col}_ab_swap' for col in PROB_COLUMNS]].to_numpy()
tuned_word_char_probs = merged_valid[[f'{col}_word_char' for col in PROB_COLUMNS]].to_numpy()

weight_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ensemble_results = []

for weight_ab_swap in weight_list:
    weight_word_char = 1 - weight_ab_swap
    ensemble_probs = weight_ab_swap * ab_swap_probs + weight_word_char * tuned_word_char_probs
    metrics = evaluate_probs(ensemble_y_valid, ensemble_probs)
    metrics['weight_ab_swap'] = weight_ab_swap
    metrics['weight_word_char'] = weight_word_char
    ensemble_results.append(metrics)
    print(
        f"weight_ab_swap={weight_ab_swap:.1f}: "
        f"log_loss={metrics['valid_log_loss']:.6f}, "
        f"accuracy={metrics['valid_accuracy']:.6f}, "
        f"macro_f1={metrics['valid_macro_f1']:.6f}"
    )

ensemble_results_df = pd.DataFrame(ensemble_results).sort_values('valid_log_loss').reset_index(drop=True)
ensemble_results_df.to_csv(ensemble_results_path, index=False, encoding='utf-8-sig')

best_ensemble_row = ensemble_results_df.iloc[0]
best_ensemble_weight_ab_swap = float(best_ensemble_row['weight_ab_swap'])
best_ensemble_weight_word_char = float(best_ensemble_row['weight_word_char'])
best_ensemble_valid_log_loss = float(best_ensemble_row['valid_log_loss'])
best_ensemble_valid_accuracy = float(best_ensemble_row['valid_accuracy'])
best_ensemble_valid_macro_f1 = float(best_ensemble_row['valid_macro_f1'])

print(f'Saved ensemble results: {ensemble_results_path}')
display(ensemble_results_df)

weight_ab_swap=0.0: log_loss=1.078429, accuracy=0.395007, macro_f1=0.388729
weight_ab_swap=0.1: log_loss=1.078305, accuracy=0.397008, macro_f1=0.397823
weight_ab_swap=0.2: log_loss=1.078202, accuracy=0.397269, macro_f1=0.398085
weight_ab_swap=0.3: log_loss=1.078119, accuracy=0.397269, macro_f1=0.398150
weight_ab_swap=0.4: log_loss=1.078056, accuracy=0.395703, macro_f1=0.396522
weight_ab_swap=0.5: log_loss=1.078013, accuracy=0.396225, macro_f1=0.396999
weight_ab_swap=0.6: log_loss=1.077991, accuracy=0.397617, macro_f1=0.398426
weight_ab_swap=0.7: log_loss=1.077990, accuracy=0.396399, macro_f1=0.397240
weight_ab_swap=0.8: log_loss=1.078008, accuracy=0.396660, macro_f1=0.397498
weight_ab_swap=0.9: log_loss=1.078047, accuracy=0.395094, macro_f1=0.395974
weight_ab_swap=1.0: log_loss=1.078106, accuracy=0.395442, macro_f1=0.396300
Saved ensemble results: D:\LLM_Classification_finetuning\outputs\logs\tuned_word_char_ensemble_results.csv


,valid_log_loss,valid_accuracy,valid_macro_f1,weight_ab_swap,weight_word_char
0,1.077990,0.396399,0.397240,0.7,0.3
1,1.077991,0.397617,0.398426,0.6,0.4
2,1.078008,0.396660,0.397498,0.8,0.2
3,1.078013,0.396225,0.396999,0.5,0.5
4,1.078047,0.395094,0.395974,0.9,0.1
5,1.078056,0.395703,0.396522,0.4,0.6
6,1.078106,0.395442,0.396300,1.0,0.0
7,1.078119,0.397269,0.398150,0.3,0.7
8,1.078202,0.397269,0.398085,0.2,0.8
9,1.078305,0.397008,0.397823,0.1,0.9


## 10. Optional Final Ensemble Submission

Create the final ensemble submission only if the best tuned ensemble improves over 1.077993.

In [10]:
final_submission_created = False
final_ensemble_submission = None

if best_ensemble_valid_log_loss < ENSEMBLE_BASELINE_LOG_LOSS:
    ab_swap_submission = pd.read_csv(ab_swap_submission_path, encoding='utf-8-sig')
    tuned_submission_for_ensemble = pd.read_csv(tuned_submission_path, encoding='utf-8-sig')

    merged_submission = ab_swap_submission[SUB_COLUMNS].merge(
        tuned_submission_for_ensemble[SUB_COLUMNS],
        on='id',
        suffixes=('_ab_swap', '_word_char'),
        how='inner',
    )
    if len(merged_submission) != len(ab_swap_submission) or len(merged_submission) != len(tuned_submission_for_ensemble):
        raise ValueError('Submission files are not perfectly aligned by id.')

    ab_submission_probs = merged_submission[[f'{col}_ab_swap' for col in SUB_PROB_COLUMNS]].to_numpy()
    word_char_submission_probs = merged_submission[[f'{col}_word_char' for col in SUB_PROB_COLUMNS]].to_numpy()
    final_submission_probs = (
        best_ensemble_weight_ab_swap * ab_submission_probs
        + best_ensemble_weight_word_char * word_char_submission_probs
    )
    final_ensemble_submission = pd.DataFrame({
        'id': merged_submission['id'],
        'winner_model_a': final_submission_probs[:, 0],
        'winner_model_b': final_submission_probs[:, 1],
        'winner_tie': final_submission_probs[:, 2],
    })
    final_ensemble_submission.to_csv(ensemble_submission_path, index=False, encoding='utf-8-sig')
    final_submission_created = True
    print(f'Saved final ensemble submission: {ensemble_submission_path}')
    print(f'probability sums close to 1: {np.allclose(final_ensemble_submission[SUB_PROB_COLUMNS].sum(axis=1), 1.0, atol=1e-6)}')
    print(f'has NaN: {final_ensemble_submission.isna().any().any()}')
    display(final_ensemble_submission.head())
else:
    print('Best tuned ensemble did not improve over 1.077993. Final ensemble submission was not created.')

Saved final ensemble submission: D:\LLM_Classification_finetuning\outputs\submissions\ensemble_ab_swap_tuned_word_char_submission.csv
probability sums close to 1: True
has NaN: False


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.244242,0.244242,0.511516
1,211333,0.350676,0.350051,0.299272
2,1233961,0.394433,0.394433,0.211135


## 11. Save Experiment Result

Append the best tuned ensemble result to `experiment_results.csv`.

In [11]:
experiment_result = pd.DataFrame([
    {
        'model_name': 'ensemble_ab_swap_tuned_word_char',
        'valid_log_loss': best_ensemble_valid_log_loss,
        'valid_accuracy': best_ensemble_valid_accuracy,
        'valid_macro_f1': best_ensemble_valid_macro_f1,
        'max_features': 'word_100000_char_50000',
        'ngram_range': 'word_(1,2)_char_(3,5)',
        'C': best_word_char_C,
        'random_state': RANDOM_STATE,
        'notes': 'Weighted probability ensemble of A/B swap TF-IDF model and tuned word+char TF-IDF model.',
        'best_word_char_C': best_word_char_C,
        'best_word_char_valid_log_loss': best_word_char_valid_log_loss,
        'weight_ab_swap': best_ensemble_weight_ab_swap,
        'weight_word_char': best_ensemble_weight_word_char,
        'submission_created': final_submission_created,
    }
])

if experiment_results_path.exists():
    previous_results = pd.read_csv(experiment_results_path, encoding='utf-8-sig')
    experiment_results = pd.concat([previous_results, experiment_result], ignore_index=True)
else:
    experiment_results = experiment_result

experiment_results.to_csv(experiment_results_path, index=False, encoding='utf-8-sig')

print(f'Saved experiment results: {experiment_results_path}')
display(experiment_results.tail())

Saved experiment results: D:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv


,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows,train_batch_size,eval_batch_size,gradient_accumulation_steps,train_rows_original,train_rows_augmented,best_alpha,raw_valid_log_loss,weight_ab_swap,weight_word_char,submission_created,best_word_char_C,best_word_char_valid_log_loss
5,distilbert_medium,1.086948,0.371944,0.371848,NaN,NaN,NaN,42,"DistilBERT finetuning on at most 6000 samples per class, max_length=384.",distilbert-base-uncased,384.0,14400.0,3600.0,8.0,16.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,tfidf_logistic_regression_ab_swap,1.078106,0.395442,0.396300,100000.0,"(1, 2)",0.10,42,TF-IDF Logistic Regression with A/B swap augmentation. Validation set not augmented. Swap-test averaging submission also saved.,NaN,NaN,NaN,11496.0,NaN,NaN,NaN,45981.0,91962.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,tfidf_word_char_ab_swap_smooth,1.078998,0.387352,0.387994,word_100000_char_50000,"word_(1,2)_char_(3,5)",0.10,42,"TF-IDF word + char features with A/B swap augmentation, swap-test averaging, and probability smoothing.",NaN,NaN,NaN,11496.0,NaN,NaN,NaN,45981.0,91962.0,0.1,1.079283,NaN,NaN,NaN,NaN,NaN
8,ensemble_ab_swap_word_char,1.077993,0.396312,0.397224,NaN,NaN,NaN,42,Weighted probability ensemble of A/B swap TF-IDF model and word+char TF-IDF model.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7,0.3,True,NaN,NaN
9,ensemble_ab_swap_tuned_word_char,1.077990,0.396399,0.397240,word_100000_char_50000,"word_(1,2)_char_(3,5)",0.05,42,Weighted probability ensemble of A/B swap TF-IDF model and tuned word+char TF-IDF model.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7,0.3,True,0.05,1.078429


## 12. Final Checks

Print final metrics and output locations.

In [12]:
print(f'best_word_char_C: {best_word_char_C}')
print(f'best_word_char_valid_log_loss: {best_word_char_valid_log_loss:.6f}')
print(f'best_ensemble_weight_ab_swap: {best_ensemble_weight_ab_swap}')
print(f'best_ensemble_weight_word_char: {best_ensemble_weight_word_char}')
print(f'best_ensemble_valid_log_loss: {best_ensemble_valid_log_loss:.6f}')
print(f'best_ensemble_valid_accuracy: {best_ensemble_valid_accuracy:.6f}')
print(f'best_ensemble_valid_macro_f1: {best_ensemble_valid_macro_f1:.6f}')
print(f'final_submission_created: {final_submission_created}')

print('\nSaved files:')
for path in [c_tuning_results_path, tuned_valid_predictions_path, tuned_submission_path, ensemble_results_path, experiment_results_path]:
    print(f'{path.exists()} -> {path}')
if final_submission_created:
    print(f'{ensemble_submission_path.exists()} -> {ensemble_submission_path}')

print('Tuned word+char ensemble experiment finished successfully.')

best_word_char_C: 0.05
best_word_char_valid_log_loss: 1.078429
best_ensemble_weight_ab_swap: 0.7
best_ensemble_weight_word_char: 0.30000000000000004
best_ensemble_valid_log_loss: 1.077990
best_ensemble_valid_accuracy: 0.396399
best_ensemble_valid_macro_f1: 0.397240
final_submission_created: True

Saved files:
True -> D:\LLM_Classification_finetuning\outputs\logs\word_char_c_tuning_results.csv
True -> D:\LLM_Classification_finetuning\outputs\oof\tfidf_word_char_ab_swap_tuned_valid_predictions.csv
True -> D:\LLM_Classification_finetuning\outputs\submissions\tfidf_word_char_ab_swap_tuned_submission.csv
True -> D:\LLM_Classification_finetuning\outputs\logs\tuned_word_char_ensemble_results.csv
True -> D:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv
True -> D:\LLM_Classification_finetuning\outputs\submissions\ensemble_ab_swap_tuned_word_char_submission.csv
Tuned word+char ensemble experiment finished successfully.
